<a href="https://colab.research.google.com/github/nnekaanadu-art/contact-centre-agent-assist/blob/main/agent_assist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pandas


In [2]:
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Use the exact model name shown in AI Studio's model list
MODEL = "gemini-3.5-flash-lite"

In [3]:
import pandas as pd
from google.colab import files

files.upload()  # choose agent_assist_transcripts.csv
df = pd.read_csv("agent_assist_transcripts.csv")
df.head()

Saving agent_assist_transcripts.csv to agent_assist_transcripts.csv


,transcript_id,transcript,expected_intent,expected_complaint,expected_vulnerability,expected_next_best_action
0,T01,"Agent: Thank you for calling Northbridge Bank,...",lost_stolen_card,No,No,Block card and order replacement
1,T02,"Agent: Northbridge Bank, this is Marcus speaki...",transaction_dispute,No,No,Open transaction dispute case
2,T03,Agent: Thank you for calling Northbridge Bank....,suspected_fraud,No,No,Escalate to fraud team
3,T04,"Agent: Northbridge Bank, this is Liam.\nCustom...",fee_reversal,Yes,No,Apply fee reversal per policy
4,T05,"Agent: Thank you for calling Northbridge Bank,...",online_banking_access,No,No,Reset online banking access


In [4]:
PROMPT_VERSION = "v2"
SYSTEM_PROMPT = """
You are an assistant for contact centre agents at a Canadian bank.
Read the call transcript and return only a JSON object with these keys:

"summary": three sentences describing the reason for the call, what happened, and what is still open.
"intent": one of lost_stolen_card, transaction_dispute, suspected_fraud, fee_reversal, online_banking_access, mortgage_payment_change, address_update, credit_limit_increase, account_closure, bereavement, balance_inquiry.
"complaint": "Yes" or "No".
"vulnerability": "Yes" or "No".
"next_best_action": one of "Block card and order replacement", "Open transaction dispute case", "Escalate to fraud team", "Apply fee reversal per policy", "Reset online banking access", "Transfer to mortgage specialist", "Update address after verification", "Submit credit limit request", "Offer retention review before closure", "Refer to bereavement support team", "Provide information only".

Definitions:
"complaint" is "Yes" when the customer expresses dissatisfaction with the bank itself (its fees, service, or errors) or asks for their concern to be recorded. Frustration with a merchant alone is not a complaint against the bank.
"vulnerability" is "Yes" when the customer mentions something that may reduce their ability to manage their finances: bereavement, illness, age related confusion, job loss, or being pressured or manipulated by a scammer in a way that affects their ability to protect themselves. A customer who spots fraud or theft and reports it promptly is not vulnerable on that basis alone.

Return JSON only, with no other text.
"""

In [ ]:
import json, time

def analyze(transcript):
    response = client.models.generate_content(
        model=MODEL,
        contents=transcript,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json",
            http_options=types.HttpOptions(timeout=60000),
        ),
    )
    return json.loads(response.text)

results = []
for _, row in df.iterrows():
    try:
        output = analyze(row["transcript"])
        print(row["transcript_id"], "done")
    except Exception as error:
        output = {"error": str(error)}
        print(row["transcript_id"], "ERROR:", str(error)[:200])
    output["transcript_id"] = row["transcript_id"]
    results.append(output)
    time.sleep(15)  # slower pace to stay under free tier limits

results_df = pd.DataFrame(results)
results_df.head()

Note: The free Gemini tier sometimes returns a 503 (service busy) error. Any transcripts that failed were retried in a later cell, and all 20 completed successfully.

In [21]:
FIELDS = {
    "intent": "expected_intent",
    "complaint": "expected_complaint",
    "vulnerability": "expected_vulnerability",
    "next_best_action": "expected_next_best_action",
}

scored = df.merge(results_df, on="transcript_id", how="left")
for model_col, key_col in FIELDS.items():
    scored[model_col + "_correct"] = (
        scored[model_col].astype(str).str.strip().str.lower()
        == scored[key_col].astype(str).str.strip().str.lower()
    )

print("Prompt version:", PROMPT_VERSION)
for model_col in FIELDS:
    accuracy = scored[model_col + "_correct"].mean() * 100
    print(f"{model_col}: {accuracy:.0f}%")
all_correct = scored[[c + "_correct" for c in FIELDS]].all(axis=1).mean() * 100
print(f"All four fields correct: {all_correct:.0f}%")

Prompt version: v2
intent: 100%
complaint: 100%
vulnerability: 95%
next_best_action: 100%
All four fields correct: 95%


In [22]:
for model_col, key_col in FIELDS.items():
    misses = scored[~scored[model_col + "_correct"]]
    for _, row in misses.iterrows():
        print(f"{row['transcript_id']} | {model_col} | expected: {row[key_col]} | got: {row[model_col]}")

T03 | vulnerability | expected: No | got: Yes


In [23]:
scored.to_csv(f"results_{PROMPT_VERSION}.csv", index=False)
files.download(f"results_{PROMPT_VERSION}.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
FIELDS = {
    "intent": "expected_intent",
    "complaint": "expected_complaint",
    "vulnerability": "expected_vulnerability",
    "next_best_action": "expected_next_best_action",
}

scored = df.merge(results_df, on="transcript_id", how="left")
for model_col, key_col in FIELDS.items():
    scored[model_col + "_correct"] = (
        scored[model_col].astype(str).str.strip().str.lower()
        == scored[key_col].astype(str).str.strip().str.lower()
    )

print("Prompt version:", PROMPT_VERSION)
for model_col in FIELDS:
    accuracy = scored[model_col + "_correct"].mean() * 100
    print(f"{model_col}: {accuracy:.0f}%")
all_correct = scored[[c + "_correct" for c in FIELDS]].all(axis=1).mean() * 100
print(f"All four fields correct: {all_correct:.0f}%")

Prompt version: v2
intent: 75%
complaint: 75%
vulnerability: 70%
next_best_action: 75%
All four fields correct: 70%


In [13]:
for i, r in enumerate(results):
    if "error" in r:
        row = df[df["transcript_id"] == r["transcript_id"]].iloc[0]
        try:
            output = analyze(row["transcript"])
            output["transcript_id"] = r["transcript_id"]
            results[i] = output
            print(r["transcript_id"], "done on retry")
        except Exception as error:
            print(r["transcript_id"], "ERROR again:", str(error)[:200])
        time.sleep(15)

results_df = pd.DataFrame(results)
results_df.head()

T07 ERROR again: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
T20 done on retry


,summary,intent,complaint,vulnerability,next_best_action,transcript_id,error
0,The customer called to report a lost debit car...,lost_stolen_card,No,No,Block card and order replacement,T01,NaN
1,The customer called because a gym charged thei...,transaction_dispute,No,No,Open transaction dispute case,T02,NaN
2,"The customer called to report a fraudulent $1,...",suspected_fraud,No,Yes,Escalate to fraud team,T03,NaN
3,The customer called to request a fee reversal ...,fee_reversal,Yes,No,Apply fee reversal per policy,T04,NaN
4,The customer called because they were locked o...,online_banking_access,No,No,Reset online banking access,T05,NaN


In [18]:
for r in results:
    print(r["transcript_id"], "ERROR" if "error" in r else "OK")

T01 OK
T02 OK
T03 OK
T04 OK
T05 OK
T06 OK
T07 OK
T08 OK
T09 OK
T10 OK
T11 OK
T12 OK
T13 OK
T14 OK
T15 OK
T16 OK
T17 OK
T18 OK
T19 OK
T20 OK


In [17]:
for i, r in enumerate(results):
    if "error" in r:
        row = df[df["transcript_id"] == r["transcript_id"]].iloc[0]
        try:
            output = analyze(row["transcript"])
            output["transcript_id"] = r["transcript_id"]
            results[i] = output
            print(r["transcript_id"], "done on retry")
        except Exception as error:
            print(r["transcript_id"], "ERROR again:", str(error)[:200])
        time.sleep(15)

results_df = pd.DataFrame(results)
results_df.head()

T07 done on retry


,summary,intent,complaint,vulnerability,next_best_action,transcript_id
0,The customer called to report a lost debit car...,lost_stolen_card,No,No,Block card and order replacement,T01
1,The customer called because a gym charged thei...,transaction_dispute,No,No,Open transaction dispute case,T02
2,"The customer called to report a fraudulent $1,...",suspected_fraud,No,Yes,Escalate to fraud team,T03
3,The customer called to request a fee reversal ...,fee_reversal,Yes,No,Apply fee reversal per policy,T04
4,The customer called because they were locked o...,online_banking_access,No,No,Reset online banking access,T05
